# Stage 2 · RLHF Pipeline — SOLUTION
### Topics: Bradley-Terry · Reward Model Training · KL-Constrained RL · PPO-for-LLMs · Reward Hacking

> The canonical reference: InstructGPT (Ouyang et al., 2022).


In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
from typing import List, Tuple, Optional, Dict
from dataclasses import dataclass


---
## 1 · Bradley-Terry Preference Model

### Why not just collect scalar reward labels?
Human labellers are inconsistent at giving absolute scores, but reliable at **pairwise comparisons**:  
*"Which of these two completions is better?"*

### Bradley-Terry model
Given two completions $y_w$ (winner) and $y_l$ (loser), the probability of the human preferring $y_w$ is:

$$P(y_w \succ y_l | x) = \sigma(r(x, y_w) - r(x, y_l))$$

where $r(x, y)$ is a scalar reward and $\sigma$ is the sigmoid.

### Reward model training loss (negative log-likelihood)
$$\mathcal{L}_{RM} = -\mathbb{E}_{(x, y_w, y_l)}\left[\log \sigma(r(x, y_w) - r(x, y_l))\right]$$

**Key properties:**
- Only the *difference* in rewards matters, not absolute scale
- Equivalent to binary cross-entropy where label is always "chosen > rejected"
- Reward model is initialised from the SFT checkpoint (same backbone as the policy)


In [15]:
def bradley_terry_loss(
    reward_chosen:   torch.Tensor,  # (B,) — r(x, y_w) for each pair
    reward_rejected: torch.Tensor,  # (B,) — r(x, y_l) for each pair
) -> torch.Tensor:
    """
    BT loss = -mean(log σ(r_w - r_l))
             = mean(log(1 + exp(r_l - r_w)))
             = mean(softplus(r_l - r_w))
    Using log_sigmoid for numerical stability.
    """
    return -F.logsigmoid(reward_chosen - reward_rejected).mean()


def reward_model_accuracy(
    reward_chosen:   torch.Tensor,
    reward_rejected: torch.Tensor,
) -> float:
    """Fraction of pairs where r_chosen > r_rejected (should approach 1 during training)."""
    return (reward_chosen > reward_rejected).float().mean().item()


class RewardModel(nn.Module):
    """
    Reward model: takes (prompt, completion) embedding → scalar reward.
    In practice this is a LM with a linear head on the last token.
    Here we use a toy MLP over a concatenated (prompt, completion) embedding.
    """
    def __init__(self, embed_dim: int = 32, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim * 2, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),        nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, prompt_emb: torch.Tensor, completion_emb: torch.Tensor) -> torch.Tensor:
        """Returns scalar reward (B,) for each (prompt, completion) pair."""
        x = torch.cat([prompt_emb, completion_emb], dim=-1)
        return self.net(x).squeeze(-1)


# ── Sanity checks ─────────────────────────────────────────────────────────
torch.manual_seed(0)
B, D = 8, 32

# Equal rewards → loss = log(2) ≈ 0.693
r_eq = torch.zeros(B)
loss_eq = bradley_terry_loss(r_eq, r_eq)
assert abs(loss_eq.item() - math.log(2)) < 1e-5, f"Expected log(2), got {loss_eq.item()}"

# Perfect separation → loss ≈ 0
r_chosen   = torch.full((B,), 10.0)
r_rejected = torch.full((B,), -10.0)
loss_perf  = bradley_terry_loss(r_chosen, r_rejected)
assert loss_perf.item() < 1e-4, f"Expected ~0, got {loss_perf.item()}"

rm = RewardModel(D)
prompt_emb = torch.randn(B, D)
chosen_emb = torch.randn(B, D)
reject_emb = torch.randn(B, D)
r_c = rm(prompt_emb, chosen_emb)
r_r = rm(prompt_emb, reject_emb)
loss = bradley_terry_loss(r_c, r_r)
acc  = reward_model_accuracy(r_c, r_r)

print(f"bradley_terry_loss ✓  equal rewards loss={loss_eq.item():.4f} (=log2={math.log(2):.4f})")
print(f"RewardModel        ✓  loss={loss.item():.4f}  acc={acc:.2f}")


bradley_terry_loss ✓  equal rewards loss=0.6931 (=log2=0.6931)
RewardModel        ✓  loss=0.7149  acc=0.38


---
## 2 · Reward Model Training

The reward model is trained on a dataset of preference pairs $(x, y_w, y_l)$.

### Dataset construction
1. Collect prompts from users
2. Generate multiple completions per prompt using the SFT model
3. Have human labellers rank pairs (or use AI feedback)
4. Create binary preference pairs from rankings

### Training details (from InstructGPT)
- Initialise from SFT model (same backbone, add a linear scalar head)
- Shuffle all pairs; train with BT loss
- Use held-out accuracy as early stopping criterion
- **Important:** treat each comparison independently (not per-prompt)

### Preference dataset simulation
For testing, we simulate a "true" reward function and generate synthetic preferences:
$y_w \succ y_l \iff r_{true}(y_w) > r_{true}(y_l)$  (with occasional noise)


In [16]:
@dataclass
class PreferencePair:
    prompt_emb:   torch.Tensor   # (D,)
    chosen_emb:   torch.Tensor   # (D,)
    rejected_emb: torch.Tensor   # (D,)


def make_synthetic_dataset(
    n_pairs: int = 200,
    embed_dim: int = 32,
    noise: float = 0.1,
    seed: int = 0,
) -> List[PreferencePair]:
    """
    Synthetic preference dataset.
    True reward = dot(completion_emb, fixed_direction).
    Pairs are labeled correctly with probability 1-noise.
    """
    torch.manual_seed(seed)
    direction = F.normalize(torch.randn(embed_dim), dim=0)  # true reward direction
    pairs = []
    for _ in range(n_pairs):
        prompt_emb = torch.randn(embed_dim)
        emb_a = torch.randn(embed_dim)
        emb_b = torch.randn(embed_dim)
        r_a = (emb_a * direction).sum().item()
        r_b = (emb_b * direction).sum().item()

        # Assign chosen/rejected based on true reward (with noise)
        if r_a > r_b:
            chosen, rejected = emb_a, emb_b
        else:
            chosen, rejected = emb_b, emb_a
        if torch.rand(1).item() < noise:
            chosen, rejected = rejected, chosen   # flip with probability noise

        pairs.append(PreferencePair(prompt_emb, chosen, rejected))
    return pairs


def train_reward_model(
    rm: RewardModel,
    dataset: List[PreferencePair],
    n_epochs: int = 10,
    batch_size: int = 32,
    lr: float = 1e-3,
) -> List[float]:
    """Train RM and return per-epoch accuracy on training data."""
    optimizer = torch.optim.Adam(rm.parameters(), lr=lr)
    acc_history = []

    for epoch in range(n_epochs):
        # Shuffle
        indices = torch.randperm(len(dataset)).tolist()
        epoch_loss, epoch_acc = [], []

        for i in range(0, len(dataset), batch_size):
            batch = [dataset[j] for j in indices[i:i+batch_size]]
            prompt = torch.stack([p.prompt_emb   for p in batch])
            chosen = torch.stack([p.chosen_emb   for p in batch])
            rejected = torch.stack([p.rejected_emb for p in batch])

            r_c = rm(prompt, chosen)
            r_r = rm(prompt, rejected)
            loss = bradley_terry_loss(r_c, r_r)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss.append(loss.item())
            epoch_acc.append(reward_model_accuracy(r_c.detach(), r_r.detach()))

        avg_acc = np.mean(epoch_acc)
        acc_history.append(avg_acc)

    return acc_history


# ── Run ───────────────────────────────────────────────────────────────────
torch.manual_seed(1)
dataset = make_synthetic_dataset(n_pairs=500, embed_dim=32, noise=0.05)
rm      = RewardModel(embed_dim=32)
history = train_reward_model(rm, dataset, n_epochs=15, batch_size=64, lr=3e-3)

print(f"Reward model training:")
print(f"  Epoch  1 accuracy: {history[0]:.3f}")
print(f"  Epoch 15 accuracy: {history[-1]:.3f}")
assert history[-1] > 0.80, f"RM should achieve >80% acc, got {history[-1]:.3f}"
assert history[-1] > history[0], "Accuracy should improve during training"
print(f"  Training ✓  accuracy improved from {history[0]:.3f} → {history[-1]:.3f}")


Reward model training:
  Epoch  1 accuracy: 0.634
  Epoch 15 accuracy: 1.000
  Training ✓  accuracy improved from 0.634 → 1.000


---
## 3 · KL-Constrained RL — The Three-Model Setup

### InstructGPT architecture
```
SFT Model  (π_ref)  — frozen reference; initialisation of policy
Policy     (π_θ)    — trained via RL; starts as copy of π_ref
Reward     (r_φ)    — frozen after RM training; scores completions
```

### The full RLHF objective
$$\max_\theta \mathbb{E}_{x \sim D,\, y \sim \pi_\theta(\cdot|x)}\left[r_\phi(x,y) - \beta \log \frac{\pi_\theta(y|x)}{\pi_{ref}(y|x)}\right]$$

The KL term $\beta \cdot D_{KL}(\pi_\theta \| \pi_{ref})$ is **crucial**:
1. Prevents reward hacking — without it, the policy finds degenerate completions the RM rates highly
2. Preserves language quality — keeps the model readable and on-distribution
3. Acts as a regulariser — $\beta$ is the most important hyperparameter

### Effective reward = RM reward − KL penalty
$$r_{eff}(x,y) = r_\phi(x,y) - \beta \cdot \log \frac{\pi_\theta(y|x)}{\pi_{ref}(y|x)}$$

The policy optimises $r_{eff}$, not just $r_\phi$.


In [17]:
def kl_divergence_approx(
    logprobs_policy: torch.Tensor,  # (B,) — log π_θ(y|x)
    logprobs_ref:    torch.Tensor,  # (B,) — log π_ref(y|x), detached
) -> torch.Tensor:
    """
    Unbiased per-sequence KL approximation:
    KL(π_θ || π_ref) ≈ log π_θ(y) - log π_ref(y)    [under π_θ's distribution]
    Returns (B,) — per-sequence KL.
    """
    return logprobs_policy - logprobs_ref.detach()


def effective_reward(
    rm_reward:       torch.Tensor,  # (B,) — r_φ(x,y)
    logprobs_policy: torch.Tensor,  # (B,) — log π_θ(y|x), detached for reward purposes
    logprobs_ref:    torch.Tensor,  # (B,) — log π_ref(y|x)
    beta:            float = 0.1,
) -> torch.Tensor:
    """
    r_eff = r_φ(x,y) - β * KL(π_θ || π_ref)
    Used to compute advantages; must be detached from policy graph.
    """
    kl = kl_divergence_approx(logprobs_policy.detach(), logprobs_ref)
    return rm_reward - beta * kl


def kl_coefficient_schedule(
    step: int,
    target_kl: float = 0.01,
    current_kl: float = 0.01,
    beta: float = 0.1,
    factor: float = 1.5,
) -> float:
    """
    Adaptive KL coefficient (OpenAI InstructGPT approach).
    If KL > 1.5 * target: increase beta (penalise drift more)
    If KL < target / 1.5: decrease beta (allow more exploration)
    """
    if current_kl > target_kl * factor:
        return beta * factor
    elif current_kl < target_kl / factor:
        return beta / factor
    return beta


# ── Sanity checks ─────────────────────────────────────────────────────────
torch.manual_seed(0)
B = 8
lp_policy = torch.randn(B, requires_grad=True)
lp_ref    = torch.randn(B)
rm_reward = torch.rand(B)

# KL = 0 when policy == ref
kl_zero = kl_divergence_approx(lp_ref, lp_ref)
assert kl_zero.abs().max().item() < 1e-5

# KL > 0 on average when policy differs from ref
kl      = kl_divergence_approx(lp_policy, lp_ref)
r_eff   = effective_reward(rm_reward, lp_policy.detach(), lp_ref, beta=0.1)
assert r_eff.shape == (B,)

# Test adaptive beta
beta_up   = kl_coefficient_schedule(0, target_kl=0.01, current_kl=0.05, beta=0.1)
beta_down = kl_coefficient_schedule(0, target_kl=0.01, current_kl=0.001, beta=0.1)
assert beta_up > 0.1,   "Beta should increase when KL is too high"
assert beta_down < 0.1, "Beta should decrease when KL is too low"

print(f"kl_divergence_approx ✓  KL(same)={kl_zero.mean().item():.5f}")
print(f"effective_reward     ✓  r_eff range: [{r_eff.min().item():.3f}, {r_eff.max().item():.3f}]")
print(f"kl_coefficient       ✓  β(KL too high)={beta_up:.3f}  β(KL too low)={beta_down:.4f}")


kl_divergence_approx ✓  KL(same)=0.00000
effective_reward     ✓  r_eff range: [-0.050, 1.180]
kl_coefficient       ✓  β(KL too high)=0.150  β(KL too low)=0.0667


---
## 4 · PPO Adapted for LLMs

Standard PPO was designed for low-dimensional action spaces. Applying it to LLMs requires several adaptations.

### Key differences

| Aspect | Standard PPO | PPO for LLMs |
|---|---|---|
| Action space | Small (e.g., 4 actions) | Huge vocabulary (50k+ tokens) |
| Episode length | Hundreds of steps | Tens to hundreds of tokens |
| Reward | Per-step from env | Terminal (end of sequence) from RM |
| Value network | Separate architecture | Value head on same backbone |
| Ratio computation | Per step | Per token, averaged over sequence |

### Token-level vs sequence-level log-probs
The probability ratio is computed per-sequence (sum of per-token log-probs):

$$\rho = \exp\!\left(\sum_t \log \pi_\theta(y_t|x,y_{<t}) - \sum_t \log \pi_{\theta_{old}}(y_t|x,y_{<t})\right)$$

### Reward assignment
Most RLHF implementations assign the scalar RM reward **only to the last token**  
and propagate backwards via GAE. This makes the reward effectively per-sequence.

$$r_t = \begin{cases} r_{eff}(x,y) - \beta \cdot KL_t & t = T \\ -\beta \cdot KL_t & t < T \end{cases}$$


In [18]:
def compute_per_token_kl(
    logits_policy: torch.Tensor,   # (B, T, V)
    logits_ref:    torch.Tensor,   # (B, T, V)  detached
) -> torch.Tensor:                 # (B, T)
    """
    Per-token KL(π_θ || π_ref) computed from logits (NOT approximate — full distribution sum).
    KL_t = Σ_v π_θ(v|.) * (log π_θ(v|.) - log π_ref(v|.))
    Use torch.distributions or manual computation.
    """
    log_p = F.log_softmax(logits_policy, dim=-1)        # (B,T,V)
    log_q = F.log_softmax(logits_ref.detach(), dim=-1)  # (B,T,V)
    p     = log_p.exp()
    return (p * (log_p - log_q)).sum(dim=-1)            # (B,T)


def build_token_rewards(
    rm_reward:   torch.Tensor,     # (B,) — scalar reward for full sequence
    per_tok_kl:  torch.Tensor,     # (B, T) — KL at each token
    comp_mask:   torch.Tensor,     # (B, T) — 1 for completion tokens
    beta:        float = 0.1,
) -> torch.Tensor:                 # (B, T)
    """
    Distribute reward:
      r_t = -beta * KL_t                     for t < T (completion)
      r_T = rm_reward - beta * KL_T          for last completion token
    Prompt positions get 0.
    """
    token_rewards = -beta * per_tok_kl * comp_mask.float()   # KL penalty everywhere

    # Add RM reward to last completion token for each sequence
    last_comp = comp_mask.cumsum(dim=-1).argmax(dim=-1)       # last=1 position, leftmost of ties
    # Actually: last position where comp_mask==1
    last_tok  = (comp_mask * torch.arange(comp_mask.shape[1], device=comp_mask.device).unsqueeze(0)).argmax(dim=-1)
    for b in range(comp_mask.shape[0]):
        token_rewards[b, last_tok[b]] += rm_reward[b]

    return token_rewards


# ── Sanity checks ─────────────────────────────────────────────────────────
torch.manual_seed(0)
B, T, V = 4, 12, 50
prompt_len = 6

logits_pol = torch.randn(B, T, V)
logits_ref = torch.randn(B, T, V)
comp_mask  = torch.cat([
    torch.zeros(B, prompt_len, dtype=torch.long),
    torch.ones( B, T - prompt_len, dtype=torch.long)
], dim=1)
rm_reward = torch.rand(B)

kl_tok   = compute_per_token_kl(logits_pol, logits_ref)
assert kl_tok.shape == (B, T), f"Expected ({B},{T}), got {kl_tok.shape}"
assert (kl_tok >= 0).all(), "KL must be non-negative"
assert kl_tok.mean().item() > 0, "KL should be > 0 for different distributions"

tok_r = build_token_rewards(rm_reward, kl_tok.detach(), comp_mask, beta=0.1)
assert tok_r.shape == (B, T)
# Prompt positions should be 0
assert (tok_r[:, :prompt_len] == 0).all(), "Prompt token rewards must be 0"

print(f"compute_per_token_kl  ✓  shape={kl_tok.shape}, mean KL={kl_tok.mean().item():.4f}")
print(f"build_token_rewards   ✓  last token reward (batch 0): {tok_r[0, 11].item():.4f}")
print(f"  = RM reward {rm_reward[0].item():.4f} - β*KL {0.1*kl_tok[0,11].item():.4f}")


compute_per_token_kl  ✓  shape=torch.Size([4, 12]), mean KL=1.0667
build_token_rewards   ✓  last token reward (batch 0): 0.9017
  = RM reward 0.9909 - β*KL 0.0892


---
## 5 · Reward Hacking, Goodhart's Law & Training Diagnostics

> *"When a measure becomes a target, it ceases to be a good measure."* — Goodhart's Law

### What is reward hacking?
The RM is an imperfect proxy for human preferences. Without the KL penalty,  
the policy finds **out-of-distribution outputs** that fool the RM while being low quality:
- Repeating tokens / phrases the RM rates highly
- Generating very long or very short outputs
- Adding sycophantic phrases ("Great question!")
- Drifting to incoherent but high-RM-score text

### Empirical signals of reward hacking

| Metric | Healthy | Hacking signal |
|---|---|---|
| RM reward | Steadily increasing | Sudden spike then plateau |
| KL from ref | Slowly increasing | KL explodes (> 20 nats) |
| Entropy | Gradually decreasing | Collapse to near 0 |
| Human eval | Tracks RM reward | Diverges from RM reward |
| Perplexity under SFT | Slightly increasing | Large increase |

### Mitigation strategies
1. **KL penalty (β):** primary defence; adaptive schedule helps
2. **Reward model ensembles:** use min of ensemble predictions (pessimistic reward)
3. **RM score clipping:** `r_eff = min(r_φ, r_max)` prevents extrapolation
4. **EOS token trick:** stop generation if policy produces EOS early
5. **Iterative RM updates:** collect new human labels on RL-generated outputs and retrain RM


In [19]:
def monitor_training(
    logprobs_policy: torch.Tensor,  # (B,)
    logprobs_ref:    torch.Tensor,  # (B,)
    rm_rewards:      torch.Tensor,  # (B,)
    entropy:         torch.Tensor,  # scalar
    step:            int,
    beta:            float,
) -> Dict[str, float]:
    """Compute and return a dict of training diagnostics."""
    kl      = (logprobs_policy - logprobs_ref.detach()).mean()
    # Scalar: mean reward minus β times mean KL (cannot .item() a length-B tensor)
    r_eff   = (rm_rewards.mean() - beta * kl).item()
    return {
        "step":          step,
        "rm_reward":     rm_rewards.mean().item(),
        "kl_from_ref":   kl.item(),
        "effective_rew": r_eff,
        "entropy":       entropy.item(),
        "beta":          beta,
    }


def detect_reward_hacking(
    metrics_history: List[Dict[str, float]],
    kl_threshold:    float = 10.0,
    entropy_min:     float = 0.1,
    rm_spike_factor: float = 2.0,
) -> List[str]:
    """
    Rule-based heuristics to detect reward hacking from training metrics.
    Returns a list of warning strings (empty if training looks healthy).
    """
    warnings = []
    if not metrics_history:
        return warnings

    latest = metrics_history[-1]

    if latest["kl_from_ref"] > kl_threshold:
        warnings.append(f"KL EXPLOSION: {latest['kl_from_ref']:.2f} > {kl_threshold}")

    if latest["entropy"] < entropy_min:
        warnings.append(f"ENTROPY COLLAPSE: {latest['entropy']:.4f} < {entropy_min}")

    if len(metrics_history) > 5:
        early_rm  = np.mean([m["rm_reward"] for m in metrics_history[:5]])
        recent_rm = np.mean([m["rm_reward"] for m in metrics_history[-5:]])
        if early_rm > 0 and recent_rm > early_rm * rm_spike_factor:
            warnings.append(f"RM SPIKE: {early_rm:.3f} → {recent_rm:.3f} (×{recent_rm/early_rm:.1f})")

    return warnings


# ── Sanity checks ─────────────────────────────────────────────────────────
torch.manual_seed(0)
B = 16
lp_pol  = torch.randn(B)
lp_ref  = torch.randn(B)
rm_rew  = torch.rand(B) * 2
entropy = torch.tensor(0.8)

metrics = monitor_training(lp_pol, lp_ref, rm_rew, entropy, step=100, beta=0.1)
assert "rm_reward" in metrics and "kl_from_ref" in metrics
print("monitor_training ✓")
for k, v in metrics.items():
    print(f"  {k:18s}: {v:.4f}" if isinstance(v, float) else f"  {k:18s}: {v}")

# Test hacking detection
history_healthy = [{"rm_reward": 0.5 + i*0.01, "kl_from_ref": 0.5,
                    "entropy": 1.0 - i*0.01} for i in range(20)]
history_hacking = history_healthy[:10] + [
    {"rm_reward": 5.0, "kl_from_ref": 15.0, "entropy": 0.02} for _ in range(10)
]
w_healthy = detect_reward_hacking(history_healthy)
w_hacking = detect_reward_hacking(history_hacking)
assert len(w_healthy) == 0, f"False alarm: {w_healthy}"
assert len(w_hacking) > 0,  f"Missed reward hacking"
print(f"\ndetect_reward_hacking ✓")
print(f"  Healthy run warnings : {w_healthy}")
print(f"  Hacking run warnings : {w_hacking}")


monitor_training ✓
  step              : 100
  rm_reward         : 0.8261
  kl_from_ref       : -0.2518
  effective_rew     : 0.8513
  entropy           : 0.8000
  beta              : 0.1000

detect_reward_hacking ✓
  Healthy run warnings : []
  Hacking run warnings : ['KL EXPLOSION: 15.00 > 10.0', 'ENTROPY COLLAPSE: 0.0200 < 0.1', 'RM SPIKE: 0.520 → 5.000 (×9.6)']
